In [ ]:
!pip install -U "transformers==4.57.1" "datasets" "evaluate" "accelerate" "sentencepiece" "scikit-learn"



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 13.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import transformers
print(transformers.__version__)


4.57.1


In [ ]:
import pandas as pd
import numpy as np
import torch
import random

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

import transformers
print("Transformers version:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())


Transformers version: 4.57.1
GPU available: True


In [ ]:
from google.colab import files

print("Upload train_vntc_10topics.csv")
uploaded = files.upload()   # chọn train_vntc_10topics.csv

print("Upload test_vntc_10topics.csv")
uploaded = files.upload()   # chọn test_vntc_10topics.csv


Upload train_vntc_10topics.csv


Saving train_vntc_10topics.csv to train_vntc_10topics (1).csv
Upload test_vntc_10topics.csv


Saving test_vntc_10topics.csv to test_vntc_10topics (1).csv


In [ ]:
train_df = pd.read_csv("train_vntc_10topics.csv", encoding="utf-8")
test_df  = pd.read_csv("test_vntc_10topics.csv", encoding="utf-8")

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("\n5 dòng đầu train:")
display(train_df.head())

print("\nSố mẫu mỗi label:")
print(train_df['label'].value_counts())


Train size: 33759
Test size: 50373

5 dòng đầu train:


,text,label
0,Thành lập dự án POLICY phòng chống HIV/AIDS ở ...,Chinh tri Xa hoi
1,Hơn 16.000 khách đến vịnh Nha Trang Theo trực ...,Chinh tri Xa hoi
2,TPHCM: Khai trương dịch vụ lặn biển săn cá mập...,Chinh tri Xa hoi
3,Du lịch VN sẽ có tư vấn nước ngoài Ông Phạm Từ...,Chinh tri Xa hoi
4,Quy chế tuyển sinh 2006: Không làm tròn điểm t...,Chinh tri Xa hoi



Số mẫu mỗi label:
label
The thao            5298
Chinh tri Xa hoi    5219
Phap luat           3868
Suc khoe            3384
Doi song            3159
Van hoa             3080
The gioi            2898
Kinh doanh          2552
Vi tinh             2481
Khoa hoc            1820
Name: count, dtype: int64


In [ ]:
train_df, valid_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df['label']
)

print("Train:", len(train_df), "Validation:", len(valid_df), "Test:", len(test_df))


Train: 30383 Validation: 3376 Test: 50373


In [ ]:
# 1. Tạo Dataset cho HuggingFace
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_ds = Dataset.from_pandas(valid_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

raw_datasets = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds
})

# 2. Mapping label <-> id
labels_list = sorted(set(train_df['label'].tolist()))
num_labels = len(labels_list)
print("Số lớp:", num_labels)
print("Danh sách nhãn:", labels_list)

label2id = {label: i for i, label in enumerate(labels_list)}
id2label = {i: label for label, i in label2id.items()}
print("\nlabel2id:", label2id)

# 3. Thêm cột label_id vào Dataset
def encode_labels(example):
    example["label_id"] = label2id[example["label"]]
    return example

raw_datasets = raw_datasets.map(encode_labels)
raw_datasets


Số lớp: 10
Danh sách nhãn: ['Chinh tri Xa hoi', 'Doi song', 'Khoa hoc', 'Kinh doanh', 'Phap luat', 'Suc khoe', 'The gioi', 'The thao', 'Van hoa', 'Vi tinh']

label2id: {'Chinh tri Xa hoi': 0, 'Doi song': 1, 'Khoa hoc': 2, 'Kinh doanh': 3, 'Phap luat': 4, 'Suc khoe': 5, 'The gioi': 6, 'The thao': 7, 'Van hoa': 8, 'Vi tinh': 9}


Map:   0%|          | 0/30383 [00:00<?, ? examples/s]

Map:   0%|          | 0/3376 [00:00<?, ? examples/s]

Map:   0%|          | 0/50373 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_id'],
        num_rows: 30383
    })
    validation: Dataset({
        features: ['text', 'label', 'label_id'],
        num_rows: 3376
    })
    test: Dataset({
        features: ['text', 'label', 'label_id'],
        num_rows: 50373
    })
})

In [ ]:
# 1. Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# 2. Hàm tokenize
def tokenize_function(examples, tokenizer, max_length=256):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

# 3. Hàm tính metric cho Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1_macro}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def train_and_evaluate_phobert(raw_datasets, num_labels, label2id, id2label,
                               model_name="phobert", model_id="vinai/phobert-base-v2"):
    print(f"\n========================")
    print(f"ĐANG TRAIN MODEL: {model_name} ({model_id})")
    print(f"========================")

    # 1. Tokenizer (dùng use_fast=False cho chắc chắn)
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)

    # 2. Tokenize dữ liệu
    tokenized = raw_datasets.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True,
        remove_columns=["text", "label"]
    )

    # Đổi tên label_id -> labels cho Trainer
    tokenized = tokenized.rename_column("label_id", "labels")
    tokenized.set_format("torch")

    # 3. Model PhoBERT
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )

    # 4. Tham số huấn luyện
    training_args = TrainingArguments(
    output_dir=f"./{model_name}-vntc",
    do_train=True,
    do_eval=True,
    eval_steps=500,
    save_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none"
)

    # 5. Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        compute_metrics=compute_metrics,
    )

    # 6. Train
    trainer.train()

    # 7. Đánh giá trên tập test
    print("\nĐánh giá trên tập TEST:")
    test_result = trainer.evaluate(tokenized["test"])
    print(test_result)

    return trainer, test_result


In [ ]:
import transformers
print(transformers.__version__)


4.57.1


In [ ]:
phobert_id = "vinai/phobert-base-v2"

phobert_trainer, phobert_test_result = train_and_evaluate_phobert(
    raw_datasets=raw_datasets,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    model_name="phobert",
    model_id=phobert_id
)

print("\nKẾT QUẢ PHOBERT TRÊN TEST:")
print(phobert_test_result)



ĐANG TRAIN MODEL: phobert (vinai/phobert-base-v2)


Map:   0%|          | 0/30383 [00:00<?, ? examples/s]

Map:   0%|          | 0/3376 [00:00<?, ? examples/s]

Map:   0%|          | 0/50373 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
100,1.801800
200,1.027900
300,0.711900
400,0.589900
500,0.521400
600,0.472700
700,0.429200
800,0.443900
900,0.417500
1000,0.378500



Đánh giá trên tập TEST:


{'eval_loss': 0.43431583046913147, 'eval_accuracy': 0.9067953070097076, 'eval_f1_macro': 0.8733053454295094, 'eval_runtime': 678.1412, 'eval_samples_per_second': 74.281, 'eval_steps_per_second': 2.323, 'epoch': 3.0}

KẾT QUẢ PHOBERT TRÊN TEST:
{'eval_loss': 0.43431583046913147, 'eval_accuracy': 0.9067953070097076, 'eval_f1_macro': 0.8733053454295094, 'eval_runtime': 678.1412, 'eval_samples_per_second': 74.281, 'eval_steps_per_second': 2.323, 'epoch': 3.0}
